# 01a: MLB Statcast Data Collection & Upload to Databricks (Batch / Full Season)

This notebook performs a **full batch load** of MLB Statcast data for one or more complete seasons. Use this notebook for:
- **Initial data population** — first-time setup of all tables
- **Full season reloads** — re-pulling an entire season's worth of data (overwrites existing data for that season)

> **For incremental (daily/weekly) updates**, use `01b_collect_incremental_data.ipynb` instead.

### What This Notebook Does
1. Installs required dependencies (pybaseball)
2. Collects pitch-level data from MLB Statcast for the specified seasons in chunks
3. Uploads raw pitch data to Databricks as a Delta table
4. Creates player dimension tables (pitchers, batters, unified)
5. Computes normalized player vector representations for similarity matching

### Tables Created / Overwritten
- `statcast_pitches` - All pitch data with season column (~1.4M rows)
- `dim_pitchers` - Pitcher information (~1K rows)
- `dim_batters` - Batter information (~2K rows)
- `dim_players` - Unified player table (~3K rows)
- `dim_pitcher_arsenal` - Pitcher pitch types and usage
- `pitcher_vectors` - Normalized pitcher statistics for similarity matching
- `batter_vectors` - Normalized batter statistics for similarity matching

### When to Use This vs. 01b
| Scenario | Use This (01a) | Use 01b (Incremental) |
|---|---|---|
| First time setup | Yes | No |
| Add a new season | Yes | No |
| Daily/weekly refresh | No | Yes |
| Mid-season catch-up | Either | Yes (preferred) |

In [ ]:
# Install pybaseball
%pip install pybaseball scikit-learn
dbutils.library.restartPython()

In [ ]:
# Imports
import pandas as pd
from pybaseball import statcast

## Configuration

Load catalog, schema, and seasons from the config file created by `00_setup`.

In [ ]:
# Load configuration from setup notebook
import json
from pathlib import Path

CONFIG = json.loads(Path("config/atbat_assistant.json").read_text())

catalog = CONFIG["workspace"]["catalog"]
schema = CONFIG["workspace"]["schema"]
seasons = CONFIG["data_collection"]["seasons"]

# Databricks compute configuration
# Option 1: Use serverless (recommended if available)
use_serverless = True

# Option 2: Use specific cluster (if serverless not available)
cluster_id = None  # Set to your cluster ID if not using serverless

print(f"Target: {catalog}.{schema}")
print(f"Seasons: {', '.join(map(str, seasons))}")
print(f"Compute: {'Serverless' if use_serverless else f'Cluster {cluster_id}'}")

In [ ]:
# Helper functions for data collection
from pyspark.sql import functions as F

def get_season_dates(year):
    """Get start and end dates for a given MLB season."""
    season_dates = {
        2024: ('2024-03-20', '2024-09-29'),
        2025: ('2025-03-27', '2025-09-28'),
        2023: ('2023-03-30', '2023-10-01'),
        2022: ('2022-04-07', '2022-10-05'),
    }
    return season_dates.get(year, (f"{year}-03-20", f"{year}-11-05"))


def pull_and_upload_statcast_chunk(start_date, end_date, season_year, catalog, schema, table_name, is_first_chunk=False):
    """
    Pull Statcast data for a date range and immediately upload to Databricks.
    This avoids memory issues and serialization timeouts.
    """
    print(f"\n  Chunk: {start_date} to {end_date}")
    
    try:
        # Pull data for this chunk
        data = statcast(start_dt=start_date, end_dt=end_date)
        
        if data is None or len(data) == 0:
            print(f"    No data returned")
            return 0
        
        # Add season column
        data['season'] = season_year
        
        print(f"    Retrieved {len(data):,} pitches")
        
        # Immediately upload this chunk
        mode = "overwrite" if is_first_chunk else "append"
        success = upload_to_databricks(data, table_name, catalog, schema, mode=mode)
        
        if success:
            return len(data)
        else:
            print(f"    Failed to upload chunk")
            return 0
            
    except Exception as e:
        print(f"    Error: {e}")
        return 0


def collect_season_in_chunks(season_year, catalog, schema, table_name, chunk_days=14, start_from_date=None, force_append=False):
    """
    Collect a full season's data in chunks and upload incrementally.
    This prevents memory issues and serialization timeouts.
    
    Args:
        start_from_date: Optional date string (YYYY-MM-DD) to resume from
        force_append: If True, always use append mode (never overwrite)
    """
    from datetime import datetime, timedelta
    
    start_date, end_date = get_season_dates(season_year)
    
    # If resuming from a specific date, override the start date
    if start_from_date:
        start_date = start_from_date
        print(f"\n{'='*70}")
        print(f"RESUMING {season_year} SEASON DATA COLLECTION")
        print(f"Resuming from: {start_date} to {end_date}")
        print(f"Chunk size: {chunk_days} days")
        print(f"Mode: APPEND (preserving existing data)")
        print(f"{'='*70}")
    else:
        print(f"\n{'='*70}")
        print(f"COLLECTING {season_year} SEASON DATA IN CHUNKS")
        print(f"Full season: {start_date} to {end_date}")
        print(f"Chunk size: {chunk_days} days")
        print(f"{'='*70}")
    
    # Parse dates
    current_date = datetime.strptime(start_date, '%Y-%m-%d')
    final_date = datetime.strptime(end_date, '%Y-%m-%d')
    
    total_pitches = 0
    chunk_num = 0
    is_first_chunk = not force_append  # If force_append, never treat as first chunk
    
    while current_date <= final_date:
        chunk_num += 1
        chunk_end = min(current_date + timedelta(days=chunk_days - 1), final_date)
        
        chunk_start_str = current_date.strftime('%Y-%m-%d')
        chunk_end_str = chunk_end.strftime('%Y-%m-%d')
        
        print(f"\nChunk {chunk_num}: {chunk_start_str} to {chunk_end_str}")
        
        # Pull and upload this chunk
        pitches = pull_and_upload_statcast_chunk(
            chunk_start_str, 
            chunk_end_str, 
            season_year, 
            catalog, 
            schema, 
            table_name,
            is_first_chunk=is_first_chunk
        )
        
        total_pitches += pitches
        is_first_chunk = False
        
        # Move to next chunk
        current_date = chunk_end + timedelta(days=1)
    
    print(f"\n{'='*70}")
    print(f"Completed {season_year} season: {total_pitches:,} total pitches")
    print(f"{'='*70}")
    
    return total_pitches


def upload_to_databricks(df, table_name, catalog, schema, mode="overwrite"):
    """
    Upload a pandas DataFrame to Databricks as a Delta table.
    Uses the existing spark session available in the notebook runtime.
    """
    try:
        full_table_name = f"{catalog}.{schema}.{table_name}"
        total_rows = len(df)
        
        print(f"  Uploading {total_rows:,} rows (mode={mode})...")
        
        # Convert pandas to Spark DataFrame
        spark_df = spark.createDataFrame(df)
        
        # For append mode, compare and align schemas
        if mode == "append":
            try:
                # Check if table exists and get its schema
                existing_df = spark.table(full_table_name)
                existing_schema = {field.name: field.dataType for field in existing_df.schema.fields}
                current_schema = {field.name: field.dataType for field in spark_df.schema.fields}
                
                # Compare schemas and identify mismatches
                mismatches = []
                new_columns = []
                
                for col_name in spark_df.columns:
                    if col_name in existing_schema:
                        existing_type_str = existing_schema[col_name].simpleString()
                        current_type_str = current_schema[col_name].simpleString()
                        
                        if existing_type_str != current_type_str:
                            mismatches.append({
                                'column': col_name,
                                'existing_type': existing_type_str,
                                'new_type': current_type_str
                            })
                            spark_df = spark_df.withColumn(col_name, 
                                F.col(col_name).cast(existing_schema[col_name]))
                    else:
                        new_columns.append(col_name)
                
                if mismatches:
                    print(f"  Schema mismatches detected - casting {len(mismatches)} columns:")
                    for mismatch in mismatches[:5]:
                        print(f"    - {mismatch['column']}: {mismatch['new_type']} -> {mismatch['existing_type']}")
                    if len(mismatches) > 5:
                        print(f"    ... and {len(mismatches) - 5} more")
                
                if new_columns:
                    print(f"  New columns detected: {len(new_columns)} (will be added via mergeSchema)")
                    
            except Exception as schema_error:
                if "Table or view not found" in str(schema_error) or "does not exist" in str(schema_error):
                    print(f"  First chunk - creating new table")
                else:
                    print(f"  Could not read existing schema: {schema_error}")
        
        # Write to Delta table
        (spark_df.write
            .format("delta")
            .mode(mode)
            .option("mergeSchema", "true")
            .saveAsTable(full_table_name))
        
        print(f"  Uploaded {total_rows:,} rows")
        return True
        
    except Exception as e:
        print(f"  Error uploading: {e}")
        return False


print("Helper functions loaded")

## Step 1: Collect Statcast Data
This will take 10-30 minutes depending on the number of seasons


In [ ]:
# Collect and upload data for each season in chunks
# This approach:
# 1. Pulls data in small date ranges (14 days by default)
# 2. Immediately uploads each chunk to Databricks
# 3. Appends subsequent chunks to the same table
# This avoids memory issues and serialization timeouts

# RESUME CONFIGURATION - Set these to resume from a specific date
# To resume: set resume_year and resume_from_date
# To start fresh: set both to None
resume_year = None  # Which year to resume (set to None to start all seasons from beginning)
resume_from_date = None  # Start date for resuming (only applies to resume_year)
use_append_mode = False  # Set to True to append to existing table (not overwrite first chunk)

table_name = "statcast_pitches"
total_pitches_all_seasons = 0

for year in seasons:
    # Determine if this year should resume from a specific date
    start_date = resume_from_date if (resume_year is not None and year == resume_year) else None
    
    pitches = collect_season_in_chunks(
        season_year=year,
        catalog=catalog,
        schema=schema,
        table_name=table_name,
        chunk_days=14,  # 14-day chunks for full runs
        start_from_date=start_date,  # Resume from specific date if applicable
        force_append=use_append_mode  # Force append mode (don't overwrite)
    )
    total_pitches_all_seasons += pitches

print(f"\n{'='*70}")
print(f"ALL SEASONS COMPLETE")
print(f"  Total pitches collected: {total_pitches_all_seasons:,}")
print(f"  Final table: {catalog}.{schema}.{table_name}")
print(f"{'='*70}")

## Step 2: Verify Uploaded Data

Data has been uploaded incrementally during collection. Let's verify the table.


In [ ]:
# Verify the uploaded table
# spark is already available in the notebook runtime
combined_data_spark = spark.table(f"{catalog}.{schema}.statcast_pitches")

print(f"Verified table: {catalog}.{schema}.statcast_pitches")
print(f"  - Total pitches: {combined_data_spark.count():,}")
print(f"  - Unique pitchers: {combined_data_spark.select('pitcher').distinct().count():,}")
print(f"  - Unique batters: {combined_data_spark.select('batter').distinct().count():,}")
print(f"  - Seasons: {sorted([row.season for row in combined_data_spark.select('season').distinct().collect()])}")

## Step 3: Create and Upload Player Dimension Tables


In [ ]:
# Create pitcher dimension table using playerid_reverse_lookup + inferred handedness from statcast
from pyspark.sql import functions as F
from pybaseball import playerid_reverse_lookup
import pandas as pd

# Get unique pitcher IDs (MLBAM IDs) from the statcast data  
pitcher_ids_spark = combined_data_spark.select('pitcher').distinct()
pitcher_ids_list = [int(row.pitcher) for row in pitcher_ids_spark.collect()]

print(f"Looking up names for {len(pitcher_ids_list):,} unique pitchers...")

# Get player names using playerid_reverse_lookup
pitcher_id_mapping = playerid_reverse_lookup(pitcher_ids_list, key_type='mlbam')

if pitcher_id_mapping is not None and len(pitcher_id_mapping) > 0:
    print(f"  Found {len(pitcher_id_mapping):,} pitcher ID mappings")
    
    # Select relevant columns and convert to Spark
    pitcher_names_pd = pitcher_id_mapping[['key_mlbam', 'name_first', 'name_last']].copy()
    pitcher_names_pd.rename(columns={'key_mlbam': 'player_id'}, inplace=True)
    pitcher_names_spark = spark.createDataFrame(pitcher_names_pd)
    
    # Get handedness (throws) from statcast data - use most common value per pitcher
    pitcher_handedness = (combined_data_spark
        .select('pitcher', 'p_throws')
        .filter(F.col('p_throws').isNotNull())
        .groupBy('pitcher')
        .agg(F.mode('p_throws').alias('throws'))
        .withColumnRenamed('pitcher', 'player_id')
    )
    
    # Join names with handedness
    pitcher_dim_spark = (pitcher_names_spark
        .join(pitcher_handedness, on='player_id', how='left')
    )
    
    pitcher_count = pitcher_dim_spark.count()
    print(f"\n✓ Created dimension table for {pitcher_count:,} pitchers")
    print(f"  Columns: {', '.join(pitcher_dim_spark.columns)}")
    
    # Write directly using Spark
    (pitcher_dim_spark.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{catalog}.{schema}.dim_pitchers"))
    
    print(f"✓ Uploaded dim_pitchers to Databricks")
else:
    print("✗ Failed to get pitcher ID mappings")


In [ ]:
# Create batter dimension table using playerid_reverse_lookup + inferred handedness from statcast
from pybaseball import playerid_reverse_lookup

# Get unique batter IDs (MLBAM IDs) from the statcast data
batter_ids_spark = combined_data_spark.select('batter').distinct()
batter_ids_list = [int(row.batter) for row in batter_ids_spark.collect()]

print(f"Looking up names for {len(batter_ids_list):,} unique batters...")

# Get player names using playerid_reverse_lookup
batter_id_mapping = playerid_reverse_lookup(batter_ids_list, key_type='mlbam')

if batter_id_mapping is not None and len(batter_id_mapping) > 0:
    print(f"  Found {len(batter_id_mapping):,} batter ID mappings")
    
    # Select relevant columns and convert to Spark
    batter_names_pd = batter_id_mapping[['key_mlbam', 'name_first', 'name_last']].copy()
    batter_names_pd.rename(columns={'key_mlbam': 'player_id'}, inplace=True)
    batter_names_spark = spark.createDataFrame(batter_names_pd)
    
    # Get handedness (bats) from statcast data - use most common value per batter
    # If a player bats from both sides, they'll show up as 'S' (switch hitter)
    batter_handedness = (combined_data_spark
        .select('batter', 'stand')
        .filter(F.col('stand').isNotNull())
        .groupBy('batter')
        .agg(F.collect_set('stand').alias('stances'))
        .withColumn('bats',
            F.when(F.size('stances') > 1, F.lit('S'))
            .otherwise(F.element_at('stances', 1))
        )
        .select('batter', 'bats')
        .withColumnRenamed('batter', 'player_id')
    )
    
    # Join names with handedness
    batter_dim_spark = (batter_names_spark
        .join(batter_handedness, on='player_id', how='left')
    )
    
    batter_count = batter_dim_spark.count()
    print(f"\n✓ Created dimension table for {batter_count:,} batters")
    print(f"  Columns: {', '.join(batter_dim_spark.columns)}")
    
    # Write directly using Spark
    (batter_dim_spark.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{catalog}.{schema}.dim_batters"))
    
    print(f"✓ Uploaded dim_batters to Databricks")
else:
    print("✗ Failed to get batter ID mappings")


In [ ]:
# Create unified player dimension table by stacking pitchers and batters
# Both tables now have the same schema from playerid_reverse_lookup + inferred handedness
# Add a 'player_type' column to indicate source

# Add player_type column to pitchers
pitcher_dim_with_type = pitcher_dim_spark.withColumn('player_type', F.lit('pitcher'))

# Add player_type column to batters
batter_dim_with_type = batter_dim_spark.withColumn('player_type', F.lit('batter'))

# Stack the two tables (union)
player_dim_stacked = pitcher_dim_with_type.unionByName(batter_dim_with_type, allowMissingColumns=True)

# Deduplicate by player_id and aggregate player_type
# If a player appears in both tables, combine their types
player_dim_spark = (player_dim_stacked
    .groupBy('player_id', 'name_first', 'name_last')
    .agg(
        F.collect_set('player_type').alias('player_types'),
        F.first('throws').alias('throws'),
        F.first('bats').alias('bats')
    )
    .withColumn('is_pitcher', F.array_contains(F.col('player_types'), 'pitcher'))
    .withColumn('is_batter', F.array_contains(F.col('player_types'), 'batter'))
    .withColumn('player_type', 
        F.when(F.col('is_pitcher') & F.col('is_batter'), F.lit('two-way'))
        .when(F.col('is_pitcher'), F.lit('pitcher'))
        .otherwise(F.lit('batter'))
    )
    .drop('player_types')
)

player_count = player_dim_spark.count()
pitchers_only = player_dim_spark.filter(F.col('player_type') == 'pitcher').count()
batters_only = player_dim_spark.filter(F.col('player_type') == 'batter').count()
two_way = player_dim_spark.filter(F.col('player_type') == 'two-way').count()

print(f"✓ Created unified player table with {player_count:,} unique players (stacked from pitchers & batters)")
print(f"  Columns: player_id, name_first, name_last, throws, bats, player_type, is_pitcher, is_batter")
print(f"  - Pitchers only: {pitchers_only:,}")
print(f"  - Batters only: {batters_only:,}")
print(f"  - Two-way players: {two_way:,}")

# Write directly using Spark
(player_dim_spark.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.{schema}.dim_players"))

print(f"✓ Uploaded dim_players to Databricks")


In [ ]:
## Step 4: Create Player Vector Representations

Compute normalized (0-1 scale) statistics for similarity matching


## Step 4: Create Player Vector Representations

Compute normalized (0-1 scale) statistics for similarity matching


In [ ]:
# Create pitcher vector representations using Spark
# IMPROVEMENTS:
# 1. Filter to >= 25 examples per pitch type
# 2. Clip outliers to 1st & 99th percentiles before aggregation (winsorization)
# 3. Clip values to [0, 1] during min-max scaling
# 4. Include pitcher handedness (p_throws)
# 5. Create separate tables for median and mean
# 6. Create vector column for embedding search

from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, DoubleType

# Select pitching-related numeric columns
pitcher_stat_cols = [
    'release_speed', 'release_spin_rate', 
    'release_pos_x', 'release_pos_y', 'release_pos_z',
    'release_extension',
    'pfx_x', 'pfx_z',
    'vx0', 'vy0', 'vz0',
    'ax', 'ay', 'az',
    'effective_speed', 'arm_angle'
]

# Filter to columns that exist
available_cols = combined_data_spark.columns
pitcher_stat_cols = [col for col in pitcher_stat_cols if col in available_cols]

print(f"Computing pitcher vectors with {len(pitcher_stat_cols)} features:")
print(f"  {', '.join(pitcher_stat_cols)}")

# Step 1: Filter to pitcher-season-pitch_type combinations with >= 25 pitches
# Group by pitcher, season, pitch_type only (p_throws should be consistent per pitcher)
pitcher_filtered = (combined_data_spark
    .groupBy('pitcher', 'season', 'pitch_type')
    .agg(
        F.count('*').alias('pitch_count'),
        F.first('p_throws').alias('p_throws')  # Get the handedness
    )
    .filter(F.col('pitch_count') >= 25)
)

print(f"\nPitcher-season-pitch_type combinations with >= 25 pitches: {pitcher_filtered.count():,}")

# Step 2: Join back to get only those pitches
pitcher_data_filtered = combined_data_spark.join(
    pitcher_filtered.select('pitcher', 'season', 'pitch_type'),
    on=['pitcher', 'season', 'pitch_type'],
    how='inner'
)

# For each pitcher-season-pitch_type, calculate percentiles and filter
def create_pitcher_vectors_with_percentile_filter(stat_type='median'):
    """Create pitcher vectors using median or mean, with percentile filtering"""
    
    # Build aggregation for percentile bounds (1st and 99th percentile)
    percentile_exprs = []
    for col_name in pitcher_stat_cols:
        percentile_exprs.append(F.expr(f"percentile_approx({col_name}, 0.01)").alias(f"{col_name}_p01"))
        percentile_exprs.append(F.expr(f"percentile_approx({col_name}, 0.99)").alias(f"{col_name}_p99"))
    
    # Get percentile bounds for each group
    percentile_bounds = (pitcher_data_filtered
        .groupBy('pitcher', 'season', 'pitch_type')
        .agg(*percentile_exprs, 
             F.count('*').alias('pitch_count_bounds'))
    )
    
    # Join bounds back (don't include p_throws in the join to avoid ambiguity)
    pitcher_with_bounds = pitcher_data_filtered.join(
        percentile_bounds,
        on=['pitcher', 'season', 'pitch_type'],
        how='inner'
    )
    
    # Clip values to 1st-99th percentile range (not filter, but clip)
    for col_name in pitcher_stat_cols:
        pitcher_with_bounds = pitcher_with_bounds.withColumn(
            col_name,
            F.when(F.col(col_name).isNull(), None)
            .when(F.col(col_name) < F.col(f"{col_name}_p01"), F.col(f"{col_name}_p01"))
            .when(F.col(col_name) > F.col(f"{col_name}_p99"), F.col(f"{col_name}_p99"))
            .otherwise(F.col(col_name))
        )
    
    # Now aggregate using median or mean
    agg_exprs = []
    for col_name in pitcher_stat_cols:
        if stat_type == 'median':
            agg_exprs.append(F.expr(f"percentile_approx({col_name}, 0.5)").alias(col_name))
        else:  # mean
            agg_exprs.append(F.mean(col_name).alias(col_name))
    
    agg_exprs.append(F.count('*').alias('pitch_count'))
    agg_exprs.append(F.first('p_throws').alias('p_throws'))
    
    pitcher_agg = (pitcher_with_bounds
        .groupBy('pitcher', 'season', 'pitch_type')
        .agg(*agg_exprs)
        .withColumnRenamed('pitcher', 'player_id')
    )
    
    print(f"\n{stat_type.upper()} vectors after percentile filtering: {pitcher_agg.count():,} rows")
    
    # Step 3: Min-max normalization with clipping to [0, 1]
    # Calculate global min/max for each feature
    feature_bounds = {}
    for col_name in pitcher_stat_cols:
        bounds = pitcher_agg.agg(
            F.min(col_name).alias('min_val'),
            F.max(col_name).alias('max_val')
        ).first()
        feature_bounds[col_name] = (bounds['min_val'], bounds['max_val'])
    
    # Normalize and clip
    for col_name in pitcher_stat_cols:
        min_val, max_val = feature_bounds[col_name]
        
        if min_val is not None and max_val is not None and max_val != min_val:
            # Normalize: (x - min) / (max - min), then clip to [0, 1]
            pitcher_agg = pitcher_agg.withColumn(
                col_name,
                F.when(F.col(col_name).isNull(), 0.0)
                .otherwise(
                    F.when((F.col(col_name) - min_val) / (max_val - min_val) < 0, 0.0)
                    .when((F.col(col_name) - min_val) / (max_val - min_val) > 1, 1.0)
                    .otherwise((F.col(col_name) - min_val) / (max_val - min_val))
                )
            )
        else:
            pitcher_agg = pitcher_agg.withColumn(col_name, F.lit(0.0))
    
    # Step 4: Keep handedness (p_throws) as a filter column, not in embedding
    
    # Step 6: Create vector column with only the statistical features (no handedness)
    vector_cols = pitcher_stat_cols
    pitcher_agg = pitcher_agg.withColumn(
        'embedding_vector',
        F.array(*[F.coalesce(F.col(c), F.lit(0.0)) for c in vector_cols])
    )
    
    # Select final columns
    pitcher_agg = pitcher_agg.select(
        'player_id', 'season', 'pitch_type', 'p_throws', 'pitch_count',
        *pitcher_stat_cols, 'embedding_vector'
    )
    
    return pitcher_agg

# Create both median and mean tables
pitcher_vectors_median = create_pitcher_vectors_with_percentile_filter('median')
pitcher_vectors_mean = create_pitcher_vectors_with_percentile_filter('mean')

# Join with dim_pitchers to add player names
pitcher_names = spark.table(f"{catalog}.{schema}.dim_pitchers").select('player_id', 'name_first', 'name_last')
pitcher_vectors_median = pitcher_vectors_median.join(pitcher_names, on='player_id', how='left')
pitcher_vectors_mean = pitcher_vectors_mean.join(pitcher_names, on='player_id', how='left')

# Write median table
(pitcher_vectors_median.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.{schema}.pitcher_vectors_median"))

print(f"✓ Uploaded pitcher_vectors_median to Databricks")

# Write mean table
(pitcher_vectors_mean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.{schema}.pitcher_vectors_mean"))

print(f"✓ Uploaded pitcher_vectors_mean to Databricks")

# Show sample
print(f"\nSample pitcher vector (median):")
pitcher_vectors_median.select('player_id', 'season', 'pitch_type', 'p_throws', 'embedding_vector').show(3, truncate=False)

# Create pitcher arsenal table - one row per pitcher per pitch type using Spark
print("\nCreating pitcher arsenal dimension table (using Spark)...")

# Aggregate by pitcher and pitch type
pitcher_arsenal_spark = (combined_data_spark
    .groupBy('pitcher', 'pitch_type')
    .agg(
        F.count('*').alias('pitch_count'),
        F.mean('release_speed').alias('avg_velocity'),
        F.mean('release_spin_rate').alias('avg_spin_rate'),
        F.min('season').alias('first_season'),
        F.max('season').alias('last_season')
    )
    .withColumnRenamed('pitcher', 'player_id')
)

# Calculate total pitches per pitcher for usage percentage
pitcher_totals_spark = (combined_data_spark
    .groupBy('pitcher')
    .agg(F.count('*').alias('total_pitches'))
    .withColumnRenamed('pitcher', 'player_id')
)

# Join and calculate usage percentage
pitcher_arsenal_spark = (pitcher_arsenal_spark
    .join(pitcher_totals_spark, on='player_id', how='left')
    .withColumn('usage_pct', F.round((F.col('pitch_count') / F.col('total_pitches')) * 100, 1))
)

# Add pitch name mapping using Spark when/otherwise
pitch_name_expr = (
    F.when(F.col('pitch_type') == 'FF', '4-Seam Fastball')
    .when(F.col('pitch_type') == 'SI', 'Sinker')
    .when(F.col('pitch_type') == 'FC', 'Cutter')
    .when(F.col('pitch_type') == 'SL', 'Slider')
    .when(F.col('pitch_type') == 'CH', 'Changeup')
    .when(F.col('pitch_type') == 'CU', 'Curveball')
    .when(F.col('pitch_type') == 'FS', 'Splitter')
    .when(F.col('pitch_type') == 'KC', 'Knuckle Curve')
    .when(F.col('pitch_type') == 'KN', 'Knuckleball')
    .when(F.col('pitch_type') == 'EP', 'Eephus')
    .when(F.col('pitch_type') == 'FO', 'Forkball')
    .when(F.col('pitch_type') == 'SC', 'Screwball')
    .when(F.col('pitch_type') == 'ST', 'Sweeper')
    .otherwise('Unknown')
)

pitcher_arsenal_spark = pitcher_arsenal_spark.withColumn('pitch_name', pitch_name_expr)

# Reorder columns
pitcher_arsenal_spark = pitcher_arsenal_spark.select(
    'player_id', 'pitch_type', 'pitch_name', 'pitch_count', 'usage_pct',
    'avg_velocity', 'avg_spin_rate', 'first_season', 'last_season', 'total_pitches'
)

arsenal_count = pitcher_arsenal_spark.count()
unique_pitchers = pitcher_arsenal_spark.select('player_id').distinct().count()
unique_pitch_types = pitcher_arsenal_spark.select('pitch_type').distinct().count()

print(f"✓ Created pitcher arsenal table: {arsenal_count:,} rows")
print(f"  Unique pitchers: {unique_pitchers:,}")
print(f"  Unique pitch types: {unique_pitch_types}")

top_pitch_types = pitcher_arsenal_spark.groupBy('pitch_type', 'pitch_name').count().orderBy(F.desc('count')).limit(5).collect()
print(f"  Most common pitch types:")
for row in top_pitch_types:
    print(f"    {row['pitch_type']} ({row['pitch_name']}): {row['count']:,} pitcher-pitch combinations")

# Write directly using Spark
(pitcher_arsenal_spark.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.{schema}.dim_pitcher_arsenal"))

print(f"✓ Uploaded dim_pitcher_arsenal to Databricks")


In [ ]:
# Create batter vector representations using Spark
# IMPROVEMENTS:
# 1. Filter to >= 25 examples per pitch type
# 2. Clip outliers to 1st & 99th percentiles before aggregation (winsorization)
# 3. Clip values to [0, 1] during min-max scaling
# 4. Include batter handedness (stand)
# 5. Create separate tables for median and mean
# 6. Create vector column for embedding search

# Select batting-related numeric columns (only contact metrics - nulls already excluded by Spark)
batter_stat_cols = [
    'launch_speed', 'launch_angle',
    'hit_distance_sc',
    'estimated_ba_using_speedangle', 'estimated_woba_using_speedangle',
    'woba_value', 'woba_denom',
    'babip_value',
    'iso_value',
    'launch_speed_angle',
    'barrel'
]

# Filter to columns that exist
available_cols = combined_data_spark.columns
batter_stat_cols = [col for col in batter_stat_cols if col in available_cols]

print(f"Computing batter vectors with {len(batter_stat_cols)} features:")
print(f"  {', '.join(batter_stat_cols)}")

# Step 1: Filter to batter-season-pitch_type combinations with >= 25 contact events
# Note: We count non-null launch_speed as a proxy for contact
# Group by batter, season, pitch_type only (stand should be consistent per batter)
batter_filtered = (combined_data_spark
    .filter(F.col('launch_speed').isNotNull())  # Only contact events
    .groupBy('batter', 'season', 'pitch_type')
    .agg(
        F.count('*').alias('contact_count'),
        F.first('stand').alias('stand')  # Get the batting stance
    )
    .filter(F.col('contact_count') >= 25)
)

print(f"\nBatter-season-pitch_type combinations with >= 25 contact events: {batter_filtered.count():,}")

# Step 2: Join back to get only those contact events
batter_data_filtered = (combined_data_spark
    .filter(F.col('launch_speed').isNotNull())  # Only contact events
    .join(
        batter_filtered.select('batter', 'season', 'pitch_type'),
        on=['batter', 'season', 'pitch_type'],
        how='inner'
    )
)

# For each batter-season-pitch_type, calculate percentiles and filter
def create_batter_vectors_with_percentile_filter(stat_type='median'):
    """Create batter vectors using median or mean, with percentile filtering"""
    
    # Build aggregation for percentile bounds (1st and 99th percentile)
    percentile_exprs = []
    for col_name in batter_stat_cols:
        percentile_exprs.append(F.expr(f"percentile_approx({col_name}, 0.01)").alias(f"{col_name}_p01"))
        percentile_exprs.append(F.expr(f"percentile_approx({col_name}, 0.99)").alias(f"{col_name}_p99"))
    
    # Get percentile bounds for each group
    percentile_bounds = (batter_data_filtered
        .groupBy('batter', 'season', 'pitch_type')
        .agg(*percentile_exprs, 
             F.count('*').alias('contact_count_bounds'))
    )
    
    # Join bounds back (don't include stand in the join to avoid ambiguity)
    batter_with_bounds = batter_data_filtered.join(
        percentile_bounds,
        on=['batter', 'season', 'pitch_type'],
        how='inner'
    )
    
    # Clip values to 1st-99th percentile range (not filter, but clip)
    for col_name in batter_stat_cols:
        batter_with_bounds = batter_with_bounds.withColumn(
            col_name,
            F.when(F.col(col_name).isNull(), None)
            .when(F.col(col_name) < F.col(f"{col_name}_p01"), F.col(f"{col_name}_p01"))
            .when(F.col(col_name) > F.col(f"{col_name}_p99"), F.col(f"{col_name}_p99"))
            .otherwise(F.col(col_name))
        )
    
    # Now aggregate using median or mean
    agg_exprs = []
    for col_name in batter_stat_cols:
        if stat_type == 'median':
            agg_exprs.append(F.expr(f"percentile_approx({col_name}, 0.5)").alias(col_name))
        else:  # mean
            agg_exprs.append(F.mean(col_name).alias(col_name))
    
    agg_exprs.append(F.count('*').alias('contact_count'))
    agg_exprs.append(F.first('stand').alias('stand'))
    
    batter_agg = (batter_with_bounds
        .groupBy('batter', 'season', 'pitch_type')
        .agg(*agg_exprs)
        .withColumnRenamed('batter', 'player_id')
    )
    
    print(f"\n{stat_type.upper()} vectors after percentile filtering: {batter_agg.count():,} rows")
    
    # Step 3: Min-max normalization with clipping to [0, 1]
    # Calculate global min/max for each feature
    feature_bounds = {}
    for col_name in batter_stat_cols:
        bounds = batter_agg.agg(
            F.min(col_name).alias('min_val'),
            F.max(col_name).alias('max_val')
        ).first()
        feature_bounds[col_name] = (bounds['min_val'], bounds['max_val'])
    
    # Normalize and clip
    for col_name in batter_stat_cols:
        min_val, max_val = feature_bounds[col_name]
        
        if min_val is not None and max_val is not None and max_val != min_val:
            # Normalize: (x - min) / (max - min), then clip to [0, 1]
            batter_agg = batter_agg.withColumn(
                col_name,
                F.when(F.col(col_name).isNull(), 0.0)
                .otherwise(
                    F.when((F.col(col_name) - min_val) / (max_val - min_val) < 0, 0.0)
                    .when((F.col(col_name) - min_val) / (max_val - min_val) > 1, 1.0)
                    .otherwise((F.col(col_name) - min_val) / (max_val - min_val))
                )
            )
        else:
            batter_agg = batter_agg.withColumn(col_name, F.lit(0.0))
    
    # Step 4: Keep handedness (stand) as a filter column, not in embedding
    
    # Step 6: Create vector column with only the statistical features (no handedness)
    vector_cols = batter_stat_cols
    batter_agg = batter_agg.withColumn(
        'embedding_vector',
        F.array(*[F.coalesce(F.col(c), F.lit(0.0)) for c in vector_cols])
    )
    
    # Select final columns
    batter_agg = batter_agg.select(
        'player_id', 'season', 'pitch_type', 'stand', 'contact_count',
        *batter_stat_cols, 'embedding_vector'
    )
    
    return batter_agg

# Create both median and mean tables
batter_vectors_median = create_batter_vectors_with_percentile_filter('median')
batter_vectors_mean = create_batter_vectors_with_percentile_filter('mean')

# Join with dim_batters to add player names
batter_names = spark.table(f"{catalog}.{schema}.dim_batters").select('player_id', 'name_first', 'name_last')
batter_vectors_median = batter_vectors_median.join(batter_names, on='player_id', how='left')
batter_vectors_mean = batter_vectors_mean.join(batter_names, on='player_id', how='left')

# Write median table
(batter_vectors_median.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.{schema}.batter_vectors_median"))

print(f"✓ Uploaded batter_vectors_median to Databricks")

# Write mean table
(batter_vectors_mean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.{schema}.batter_vectors_mean"))

print(f"✓ Uploaded batter_vectors_mean to Databricks")

# Show sample
print(f"\nSample batter vector (median):")
batter_vectors_median.select('player_id', 'season', 'pitch_type', 'stand', 'embedding_vector').show(3, truncate=False)


In [ ]:
import pandas as pd
from pybaseball import batting_stats, pitching_stats
from pyspark.sql import functions as F

def _normalize_name_expr(col):
    # Lowercase, trim, remove non-letters for robust name joins
    return F.regexp_replace(F.lower(F.trim(col)), r'[^a-z]', '')

def _prepare_fg_names(df: pd.DataFrame) -> pd.DataFrame:
    # FanGraphs 'Name' is usually "Last, First". Convert to "First Last".
    # Fallback: if comma absent, leave as-is.
    def to_first_last(name: str) -> str:
        if not isinstance(name, str):
            return ''
        parts = [p.strip() for p in name.split(',')]
        if len(parts) == 2:
            return f"{parts[1]} {parts[0]}".strip()
        return name.strip()

    df = df.copy()
    df['player_name'] = df['Name'].apply(to_first_last)
    df['team'] = df['Team'].astype(str).str.strip()
    # Standardize column names
    if 'Season' not in df.columns:
        # pybaseball should include Season when fetching single season; if not, rely on the passed year
        raise ValueError("Expected 'Season' column not found in FanGraphs stats output")
    df.rename(columns={'Season': 'year'}, inplace=True)
    return df[['year', 'player_name', 'team']]

def _fetch_batting_fg(seasons_list):
    frames = []
    for y in seasons_list:
        try:
            pdf = batting_stats(int(y), qual=1)
            pdf['Season'] = int(y)
            frames.append(pdf[['Season', 'Name', 'Team']])
        except Exception as e:
            print(f"Warning: batting_stats({y}) failed: {e}")
    if not frames:
        return pd.DataFrame(columns=['year', 'player_name', 'team'])
    return _prepare_fg_names(pd.concat(frames, ignore_index=True))

def _fetch_pitching_fg(seasons_list):
    frames = []
    for y in seasons_list:
        try:
            pdf = pitching_stats(int(y), qual=1)
            pdf['Season'] = int(y)
            frames.append(pdf[['Season', 'Name', 'Team']])
        except Exception as e:
            print(f"Warning: pitching_stats({y}) failed: {e}")
    if not frames:
        return pd.DataFrame(columns=['year', 'player_name', 'team'])
    return _prepare_fg_names(pd.concat(frames, ignore_index=True))

# 1) Fetch FanGraphs data (batters and pitchers)
batters_fg_pd = _fetch_batting_fg(seasons)
pitchers_fg_pd = _fetch_pitching_fg(seasons)

# 2) Convert to Spark
batters_fg_spark = spark.createDataFrame(batters_fg_pd)
pitchers_fg_spark = spark.createDataFrame(pitchers_fg_pd)

# 3) Prepare dim_players mapping with normalized full name
dim_players = spark.table(f"{catalog}.{schema}.dim_players").select('player_id', 'name_first', 'name_last')
dim_players_named = (
    dim_players
    .withColumn('dim_player_name', F.concat_ws(' ', F.col('name_first'), F.col('name_last')))
    .withColumn('player_name_norm', _normalize_name_expr(F.col('dim_player_name')))
    .select('player_id', 'dim_player_name', 'player_name_norm')
)

# 4) Normalize FG names and join to get player_id (batters)
batters_joined = (
    batters_fg_spark
    .withColumn('player_name_norm', _normalize_name_expr(F.col('player_name')))
    .join(dim_players_named, on='player_name_norm', how='left')
    .withColumn('player_name', F.coalesce(F.col('dim_player_name'), F.col('player_name')))
    .select('year', 'player_id', 'player_name', 'team')
    .distinct()
)

pitchers_joined = (
    pitchers_fg_spark
    .withColumn('player_name_norm', _normalize_name_expr(F.col('player_name')))
    .join(dim_players_named, on='player_name_norm', how='left')
    .withColumn('player_name', F.coalesce(F.col('dim_player_name'), F.col('player_name')))
    .select('year', 'player_id', 'player_name', 'team')
    .distinct()
)

# 5) Write Delta tables
(batters_joined.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.{schema}.dim_batter_team_year"))

(pitchers_joined.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.{schema}.dim_pitcher_team_year"))

# 6) Quick verification
total_bat = batters_joined.count()
matched_bat = batters_joined.filter(F.col('player_id').isNotNull()).count()
total_pit = pitchers_joined.count()
matched_pit = pitchers_joined.filter(F.col('player_id').isNotNull()).count()

print(f"✓ dim_batter_team_year written: {total_bat:,} rows ({matched_bat:,} with player_id)")
print(f"✓ dim_pitcher_team_year written: {total_pit:,} rows ({matched_pit:,} with player_id)")